# Discrete Flow Matching for 7 × 7 Queens

Transform a fully colored puzzle into a sparse board: **queens retain their original region colors; all other cells become EMPTY**.

Exactly one queen belongs to each row, column and region; queens cannot touch diagonally. Longer diagonals are allowed.

1. Prepared dataset and forward-path visualization.
2. Online categorical flow-matching training.
3. Model saving and loading.
4. Standard stochastic DFM solution trajectories.
5. Strict solution success rate on held-out puzzles.

The dataset is stored in <code>data/queens7/</code>. Running this notebook never generates or searches for new puzzles.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import html
from collections import Counter, defaultdict
import json
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler
from _shared.model import Denoiser
from _shared.queens_data import (
    BOARD_SIZE,
    EMPTY,
    load_queens_dataset,
    solution_checks,
    split_puzzles,
)

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = BOARD_SIZE + 1
FLOW_STEPS = 100
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens7"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "queens7_discrete_flow_matching.pt"
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]
PLOT_CONFIG = {"responsive": True, "displaylogo": False, "displayModeBar": False}
display(
    HTML(
        f"<b>Queens 7×7 · Discrete Flow Matching</b><br>"
        f"Device: {html.escape(str(DEVICE))} · EMPTY token: {EMPTY} · {FLOW_STEPS} sampling steps"
    )
)

## 1. Prepared dataset

The collection combines already solved **AMAZE**, **Queens.game** and **Queendom** examples with synthetic pairs prepared offline. Every record includes its original source and queen coordinates.

- External answers are copied from their sources; their uniqueness claim is trusted.
- Synthetic boards were checked for uniqueness once during offline preparation. Synthetic regions are grown round robin, then their boundaries are refined to obtain a unique solution while preserving connectivity and 2?12 cells per region.
- Rotated, reflected and color-renamed duplicates are removed before splitting.
- Only training uses geometric augmentations and color permutations.
- We make our own split; this is not an official AMAZE benchmark.

Sources: [AMAZE data](https://huggingface.co/datasets/piekenius123/Amaze), [AMAZE construction](https://github.com/spatigen/amaze/tree/main/queen-generator), [Queens.game](https://queens.game/about.html), [Queendom](https://github.com/bonesmoses/queendom).

To rebuild outside the notebook, run <code>scripts/collect_queens7.py</code>, then <code>scripts/generate_queens7.py --count 5000</code>.


In [2]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens7")
splits = split_puzzles(puzzles["fingerprints"], seed=SEED)
regions, queen_masks = puzzles["regions"], puzzles["queens"]
target_tokens = np.where(queen_masks, regions, EMPTY).astype(np.int64)


class QueensDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        source_index = int(self.indices[index])
        board = torch.from_numpy(regions[source_index].astype(np.int64))
        queens = torch.from_numpy(queen_masks[source_index].copy())
        if self.augment:
            turns = int(torch.randint(4, ()).item())
            board = torch.rot90(board, turns, dims=(-2, -1))
            queens = torch.rot90(queens, turns, dims=(-2, -1))
            if bool(torch.randint(2, ()).item()):
                board, queens = board.flip(-1), queens.flip(-1)
            board = torch.randperm(BOARD_SIZE)[board]
        return board.contiguous(), torch.where(queens, board, EMPTY).contiguous()


train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
source_summary = " · ".join(
    f"{html.escape(name)}: {count:,}"
    for name, count in puzzles["metadata"]["source_counts"].items()
)
split_summary = " · ".join(
    f"{name}: {len(indices):,}" for name, indices in splits.items()
)
display(
    HTML(
        f"<b>{len(regions):,} distinct puzzles</b><br>{source_summary}<br>{split_summary}"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

### 1.1 Examples from every source

Draw up to two distinct examples per source before augmentation, with the supplied queens directly on the complete region coloring. Queendom contributes one example.


In [4]:
SOURCE_EXAMPLES = 2
source_example_rng = np.random.default_rng(SEED + 201)
dataset_source_examples = {}
for source_name in sorted(set(puzzles["source_names"].tolist())):
    available = np.flatnonzero(puzzles["source_names"] == source_name)
    chosen = source_example_rng.choice(
        available,
        size=min(SOURCE_EXAMPLES, len(available)),
        replace=False,
    )
    dataset_source_examples[source_name] = chosen
    fig = make_subplots(
        rows=1,
        cols=len(chosen),
        subplot_titles=[f"Example {number}" for number in range(1, len(chosen) + 1)],
        horizontal_spacing=0.07,
    )
    for col, index in enumerate(chosen, start=1):
        add_board(fig, regions[index], regions[index], 1, col)
        queen_rows, queen_cols = np.where(queen_masks[index])
        fig.add_trace(
            go.Scatter(
                x=queen_cols,
                y=queen_rows,
                mode="text",
                text=[chr(0x265B)] * len(queen_rows),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    style_boards(
        fig,
        f"{source_name} ? {len(available):,} puzzles",
        rows=1,
    ).show(config=PLOT_CONFIG)

### 1.2 Number of colorings per solution

Group the entire prepared dataset by the exact queen positions: the queen's column in each row.
Let $K(s)$ be the number of distinct region maps assigned to solution $s$. Under a uniform choice
among the distinct observed solutions,

$$
P(K=k)=\frac{\#\{s:K(s)=k\}}{\#\{\text{distinct observed solutions}\}}.
$$

This is the empirical distribution in our dataset, with no augmentation. The dataset has already
removed duplicate region maps up to rotations, reflections and color relabeling.
Here queen positions retain their exact orientation; rotated queen placements count as different solutions.


In [5]:
colorings_by_solution = defaultdict(list)
for index, record in enumerate(puzzles["records"]):
    queen_columns = tuple(column for row, column in sorted(record["queens"]))
    colorings_by_solution[queen_columns].append(index)

coloring_multiplicities = np.asarray(
    [len(indices) for indices in colorings_by_solution.values()],
    dtype=np.int64,
)
multiplicity_counts = Counter(coloring_multiplicities.tolist())
multiplicity_support = np.arange(1, int(coloring_multiplicities.max()) + 1)
multiplicity_frequencies = np.asarray(
    [multiplicity_counts.get(int(count), 0) for count in multiplicity_support]
)
multiplicity_probabilities = multiplicity_frequencies / len(colorings_by_solution)
mean_colorings = float(coloring_multiplicities.mean())
median_colorings = float(np.median(coloring_multiplicities))

fig = go.Figure(
    go.Bar(
        x=multiplicity_support,
        y=multiplicity_probabilities,
        customdata=multiplicity_frequencies,
        marker_color="#5267DB",
        hovertemplate=(
            "Colorings per solution: %{x}<br>"
            "Distinct solutions: %{customdata}<br>"
            "Probability: %{y:.2%}<extra></extra>"
        ),
    )
)
fig.add_vline(
    x=mean_colorings,
    line_color="#DC6557",
    line_width=2,
    line_dash="dash",
    annotation_text=f"Mean = {mean_colorings:.2f}",
    annotation_position="top right",
    annotation_font_color="#B85045",
)
fig.update_layout(
    title=dict(
        text=f"Colorings per solution · {len(regions):,} puzzles · "
        f"{len(colorings_by_solution):,} distinct queen placements",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=440,
    xaxis_title="Number of distinct colorings per solution",
    yaxis_title="Probability · uniformly selected distinct solution",
    font=dict(family="Arial, sans-serif", size=15),
    margin=dict(l=70, r=25, t=85, b=65),
    bargap=0.18,
)
fig.update_xaxes(
    dtick=max(1, int(np.ceil(multiplicity_support[-1] / 30))),
    range=[0.3, float(multiplicity_support[-1]) + 0.7],
)
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"<b>{len(colorings_by_solution):,} unique queen placements</b> · "
        f"mean {mean_colorings:.2f} colorings · median {median_colorings:g} · "
        f"range {coloring_multiplicities.min()}–{coloring_multiplicities.max()}<br>"
        f"More than one coloring: {np.mean(coloring_multiplicities > 1):.2%} of solutions."
    )
)

## 2. Conditional forward path

For original coloring $c$, the source is $x_0=c$. The supplied solution defines

$$
x_1^i =
\begin{cases}
c^i, & \text{if cell } i \text{ contains a queen},\\
\mathrm{EMPTY}, & \text{otherwise}.
\end{cases}
$$

The linear categorical path is

$$
p_t(x_t^i\mid c,x_1)=(1-t)\delta_{c^i}(x_t^i)+t\delta_{x_1^i}(x_t^i).
$$

Each cell independently chooses its target token with probability $t$. Queens keep their color; other cells switch to EMPTY. A single uniform switch time per cell gives a consistent visualized trajectory.

The original coloring $c$ is always available as a condition. The supplied queen mask is never an input condition.


In [6]:
def sample_conditional_path(source, target, times, generator=None):
    switches = torch.rand(source.shape, device=source.device, generator=generator)
    return torch.where(switches < times[:, None, None], target, source)


FORWARD_TIMES = np.linspace(0, 1, 6)
forward_index = int(splits["validation"][0])
forward_source, forward_target = regions[forward_index], target_tokens[forward_index]
forward_switches = np.random.default_rng(SEED + 101).random((BOARD_SIZE, BOARD_SIZE))
fig = make_subplots(
    rows=1,
    cols=len(FORWARD_TIMES),
    subplot_titles=[f"t = {time:.1f}" for time in FORWARD_TIMES],
    horizontal_spacing=0.025,
)
for col, time in enumerate(FORWARD_TIMES, start=1):
    frame = np.where(forward_switches < time, forward_target, forward_source)
    add_board(fig, frame, forward_source, 1, col, queens=time == 1)
style_boards(fig, "Forward path · fully colored puzzle → supplied solution").show(
    config=PLOT_CONFIG
)

## 3. Conditional endpoint model

Reuse the shared U-Net at resolution $7 \to 4 \to 2 \to 4 \to 7$.

- Current state: **8 one-hot channels**, including EMPTY.
- Original region coloring: **7 one-hot channels**.
- Row and column coordinates: **2 channels**.
- Input: $(B,17,7,7)$; output: $(B,8,7,7)$ endpoint logits.
- Continuous time is embedded sinusoidally and added inside each residual block.

A colored cell can end as its own color or EMPTY; an already empty cell must stay EMPTY under this path. Logits enforce this support. Row, column, region and queen-count constraints are learned, not imposed.


In [7]:
MODEL_CONFIG = dict(
    board_size=BOARD_SIZE,
    base_channels=64,
    time_dim=128,
    time_embedding_scale=100.0,
)


class QueensDFM(nn.Module):
    def __init__(
        self, board_size=7, base_channels=64, time_dim=128, time_embedding_scale=100.0
    ):
        super().__init__()
        if board_size != BOARD_SIZE:
            raise ValueError("This experiment and dataset use 7×7 boards.")
        self.time_embedding_scale = float(time_embedding_scale)
        self.network = Denoiser(
            embedding_dim=NUM_CLASSES + BOARD_SIZE + 2,
            output_dim=NUM_CLASSES,
            base_channels=base_channels,
            time_dim=time_dim,
        )
        positions = torch.linspace(-1, 1, board_size)
        rr, cc = torch.meshgrid(positions, positions, indexing="ij")
        self.register_buffer("coordinates", torch.stack((rr, cc))[None])

    def forward(self, state, coloring, times):
        state_channels = (
            F.one_hot(state.long(), NUM_CLASSES).permute(0, 3, 1, 2).float()
        )
        color_channels = (
            F.one_hot(coloring.long(), BOARD_SIZE).permute(0, 3, 1, 2).float()
        )
        inputs = torch.cat(
            (
                state_channels,
                color_channels,
                self.coordinates.expand(len(state), -1, -1, -1),
            ),
            dim=1,
        )
        logits = self.network(inputs, times * self.time_embedding_scale)
        allowed = F.one_hot(coloring.long(), NUM_CLASSES).permute(0, 3, 1, 2).bool()
        allowed = allowed & (state != EMPTY)[:, None]
        allowed[:, EMPTY] = True
        return logits.masked_fill(~allowed, -torch.inf)


model = QueensDFM(**MODEL_CONFIG).to(DEVICE)
display(
    HTML(
        f"<b>Conditional U-Net</b> · {sum(p.numel() for p in model.parameters()):,} parameters"
        " · (B, 17, 7, 7) → (B, 8, 7, 7)"
    )
)

## 4. Online training

Independently for every sample, draw an augmented puzzle/solution pair, $t\sim U[0,1)$ and per-cell path switches. Predict the endpoint from $(x_t,c,t)$ with mean pixel cross-entropy:

$$
\mathcal L=-\frac{1}{B\cdot49}
\sum_{b=1}^{B}\sum_{i=1}^{49}
\log p_\theta(x_{1,b}^i\mid x_{t,b},c_b,t_b).
$$

Known EMPTY cells contribute zero loss. The loss plot updates in place on a logarithmic scale. Validation uses fixed times and random switches; a separate $t=0$ curve evaluates the untouched input puzzle.

Run the setup cell before training or loading. To resume, load a checkpoint and then run the training loop.


In [8]:
BATCH_SIZE = 64
EPOCHS = 100  # Additional epochs for each execution of the training loop.
SAMPLES_PER_EPOCH = 4096  # Random augmented views, sampled with replacement.
LEARNING_RATE = 2e-4
PLOT_EVERY = 20

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
history = dict(
    step=[],
    train_ce=[],
    validation_step=[],
    validation_ce=[],
    validation_t0_ce=[],
    epochs_completed=0,
)
global_step = 0


def training_loss(coloring, target):
    times = torch.rand(len(coloring), device=DEVICE)
    state = sample_conditional_path(coloring, target, times)
    return F.cross_entropy(model(state, coloring, times), target)


@torch.no_grad()
def validation_losses():
    was_training = model.training
    model.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    total, count, at_zero, count_at_zero = 0.0, 0, 0.0, 0
    try:
        for coloring, target in validation_loader:
            coloring, target = coloring.to(DEVICE), target.to(DEVICE)
            for time in (0.0, 0.2, 0.4, 0.6, 0.8, 0.95):
                times = torch.full((len(coloring),), time, device=DEVICE)
                state = sample_conditional_path(coloring, target, times, generator)
                loss = F.cross_entropy(model(state, coloring, times), target).item()
                total += loss * len(coloring)
                count += len(coloring)
                if time == 0.0:
                    at_zero += loss * len(coloring)
                    count_at_zero += len(coloring)
        return total / count, at_zero / count_at_zero
    finally:
        model.train(was_training)

In [9]:
training_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=EPOCHS * len(train_loader),
    description="Training",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
training_status = widgets.HTML()
loss_plot = go.FigureWidget()
loss_plot.add_scatter(
    name="Train CE · moving mean", mode="lines", line=dict(color="#4F6BED", width=2)
)
loss_plot.add_scatter(
    name="Validation CE · fixed path",
    mode="lines+markers",
    line=dict(color="#E49B43", width=2),
    marker=dict(size=5),
)
loss_plot.add_scatter(
    name="Validation CE at t = 0",
    mode="lines+markers",
    line=dict(color="#2F9D88", width=2, dash="dot"),
    marker=dict(size=5),
)
loss_plot.update_layout(
    title="Queens DFM · endpoint cross-entropy",
    template="plotly_white",
    xaxis_title="Optimizer step",
    yaxis_title="Mean pixel CE · log scale",
    yaxis_type="log",
    autosize=True,
    height=450,
    margin=dict(l=65, r=20, t=65, b=90),
    legend=dict(orientation="h", y=-0.2, x=0),
    font=dict(family="Arial, sans-serif", size=15),
)
display(training_progress, training_status, loss_plot)


def refresh_loss_plot():
    values = np.asarray(history["train_ce"], dtype=float)
    window = min(25, len(values))
    if not window:
        return
    smooth = np.convolve(values, np.ones(window) / window, mode="valid")
    with loss_plot.batch_update():
        loss_plot.data[0].x = history["step"][window - 1 :]
        loss_plot.data[0].y = np.maximum(smooth, 1e-12)
        loss_plot.data[1].x = history["validation_step"]
        loss_plot.data[1].y = np.maximum(history["validation_ce"], 1e-12)
        loss_plot.data[2].x = history["validation_step"]
        loss_plot.data[2].y = np.maximum(history["validation_t0_ce"], 1e-12)


refresh_loss_plot()
start_epoch = int(history["epochs_completed"])
for local_epoch in range(EPOCHS):
    model.train()
    for coloring, target in train_loader:
        coloring, target = coloring.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(coloring, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        global_step += 1
        history["step"].append(global_step)
        history["train_ce"].append(float(loss.item()))
        training_progress.value += 1
        if global_step % PLOT_EVERY == 0 or len(history["step"]) == 1:
            refresh_loss_plot()
            training_status.value = (
                f"Epoch {start_epoch + local_epoch + 1} · step {global_step:,} · "
                f"train CE {history['train_ce'][-1]:.5f}"
            )
    val_ce, val_t0_ce = validation_losses()
    history["validation_step"].append(global_step)
    history["validation_ce"].append(val_ce)
    history["validation_t0_ce"].append(val_t0_ce)
    history["epochs_completed"] = start_epoch + local_epoch + 1
    refresh_loss_plot()
    training_status.value = (
        f"Epoch {history['epochs_completed']} · step {global_step:,} · "
        f"validation CE {val_ce:.5f} · t = 0 CE {val_t0_ce:.5f}"
    )
model.eval()
training_progress.bar_style = "success"

IntProgress(value=0, description='Training', layout=Layout(width='100%'), max=6400, style=ProgressStyle(descri…

HTML(value='')

FigureWidget({
    'data': [{'line': {'color': '#4F6BED', 'width': 2},
              'mode': 'lines',
              'name': 'Train CE · moving mean',
              'type': 'scatter',
              'uid': 'd9d69e48-ae53-4525-8715-6eaf57ca0223'},
             {'line': {'color': '#E49B43', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Validation CE · fixed path',
              'type': 'scatter',
              'uid': 'be7428d9-ec6e-48d4-9237-94c7deec4714'},
             {'line': {'color': '#2F9D88', 'dash': 'dot', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Validation CE at t = 0',
              'type': 'scatter',
              'uid': '465f3a2e-d5b5-4387-a0ff-882bb9e2ed7b'}],
    'layout': {'autosize': True,
               'font': {'family': 'Arial, sans-serif', 'size': 15},
               'height': 450,
               'legend': {'orientation': 'h', 'x': 0, 

## 5. Save and load

### Save

Save weights, optimizer, loss history, exact dataset fingerprint and train/validation/test split to the root <code>artifacts/queens7/</code> directory.


In [10]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "format_version": "queens7_dfm_v1",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": MODEL_CONFIG,
        "history": history,
        "global_step": global_step,
        "flow_steps": FLOW_STEPS,
        "seed": SEED,
        "dataset_sha256": puzzles["metadata"]["data_sha256"],
        "splits": {
            name: torch.as_tensor(indices.copy()) for name, indices in splits.items()
        },
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else []
        ),
    },
    CHECKPOINT_PATH,
)
display(
    HTML(
        f"Saved: <code>{html.escape(str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)))}</code>"
    )
)

### Load

Run Sections 1–3 and the training setup cell, then skip the training loop and save cell. Load here to continue directly to sampling, or run the training loop to resume. Restore the saved split and reject mismatched dataset files.


In [11]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if checkpoint["format_version"] != "queens7_dfm_v1":
    raise ValueError("Unsupported Queens checkpoint format.")
if checkpoint["dataset_sha256"] != puzzles["metadata"]["data_sha256"]:
    raise ValueError("Checkpoint and dataset differ; restore the matching dataset.")
MODEL_CONFIG = checkpoint["model_config"]
model = QueensDFM(**MODEL_CONFIG).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
history, global_step = checkpoint["history"], int(checkpoint["global_step"])
FLOW_STEPS, SEED = int(checkpoint["flow_steps"]), int(checkpoint["seed"])
splits = {name: value.numpy().copy() for name, value in checkpoint["splits"].items()}
train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
torch.set_rng_state(checkpoint["torch_rng_state"])
if DEVICE.type == "cuda" and checkpoint["cuda_rng_state_all"]:
    if len(checkpoint["cuda_rng_state_all"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])
model.eval()
display(
    HTML(
        f"<b>Loaded Queens DFM</b> · {history['epochs_completed']} epochs · "
        f"{global_step:,} optimizer steps"
    )
)

## 6. Stochastic solution trajectories

For the linear categorical path, a still-colored cell jumps to EMPTY with probability

$$
P(x_{t+\Delta t}^i=\mathrm{EMPTY}\mid x_t,c)
=\frac{\Delta t}{1-t}\,
p_\theta(x_1^i=\mathrm{EMPTY}\mid x_t,c,t).
$$

An empty cell stays empty. Every step evaluates the network once and independently samples the categorical jumps. The same puzzle always starts from its full coloring; different random seeds produce different trajectories.

This is the finite-step Euler approximation to the DFM jump process. A mistaken erasure cannot be undone. Outputs receive no backtracking, rule-based repair, top-k selection or forced count of seven queens.

Reference: [Discrete Flow Matching, §2.3–2.4](https://arxiv.org/html/2407.15595v2).


In [12]:
@torch.no_grad()
def sample_queens(
    candidate, coloring, steps=100, seed=0, capture_steps=(), progress=None
):
    if not isinstance(steps, int) or steps < 1:
        raise ValueError("steps must be a positive integer.")
    device = next(candidate.parameters()).device
    coloring = torch.as_tensor(coloring, dtype=torch.long, device=device)
    if coloring.ndim == 2:
        coloring = coloring[None]
    if coloring.ndim != 3 or tuple(coloring.shape[-2:]) != (BOARD_SIZE, BOARD_SIZE):
        raise ValueError("Expected coloring with shape (B,7,7) or (7,7).")
    if bool(((coloring < 0) | (coloring >= BOARD_SIZE)).any()):
        raise ValueError("Coloring must contain only the seven original region IDs.")
    state = coloring.clone()
    generator = torch.Generator(device=device).manual_seed(int(seed))
    capture_steps = set(map(int, capture_steps))
    if any(step < 0 or step > steps for step in capture_steps):
        raise ValueError("Capture steps must be between zero and steps.")
    snapshots = {0: state.cpu().clone()} if 0 in capture_steps else {}
    was_training = candidate.training
    candidate.eval()
    try:
        for step in range(steps):
            time = step / steps
            times = torch.full((len(state),), time, device=device)
            p_empty = candidate(state, coloring, times).softmax(dim=1)[:, EMPTY]
            # dt/(1-t) = 1/(steps-step); the final factor is exactly one.
            jump_probability = p_empty / (steps - step)
            jump = (
                torch.rand(state.shape, device=device, generator=generator)
                < jump_probability
            )
            state = torch.where(jump & (state != EMPTY), EMPTY, state)
            if step + 1 in capture_steps:
                snapshots[step + 1] = state.cpu().clone()
            if progress is not None:
                progress(step + 1, steps)
    finally:
        candidate.train(was_training)
    return state.cpu(), snapshots

In [13]:
N_TRAJECTORY_PUZZLES = 3
TRAJECTORY_ATTEMPTS = 2
CAPTURE_STEPS = tuple(sorted(set(np.rint(np.linspace(0, FLOW_STEPS, 6)).astype(int))))
trajectory_indices = splits["test"][:N_TRAJECTORY_PUZZLES]
trajectory_progress = widgets.IntProgress(
    value=0,
    max=len(trajectory_indices) * TRAJECTORY_ATTEMPTS * FLOW_STEPS,
    description="Trajectories",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(trajectory_progress)
trajectory_records = []
for puzzle_index in trajectory_indices:
    for attempt in range(TRAJECTORY_ATTEMPTS):
        offset = len(trajectory_records) * FLOW_STEPS

        def update_trajectory(done, total, offset=offset):
            trajectory_progress.value = offset + done

        final, snapshots = sample_queens(
            model,
            regions[puzzle_index],
            steps=FLOW_STEPS,
            seed=SEED + 1000 + len(trajectory_records),
            capture_steps=CAPTURE_STEPS,
            progress=update_trajectory,
        )
        solved = bool(
            solution_checks(regions[puzzle_index], final[0].numpy())["solved"]
        )
        trajectory_records.append((puzzle_index, attempt, snapshots, solved))
trajectory_progress.bar_style = "success"

for puzzle_index, attempt, snapshots, solved in trajectory_records:
    titles = [f"t = {step / FLOW_STEPS:.1f}" for step in CAPTURE_STEPS] + [
        "Supplied solution"
    ]
    fig = make_subplots(
        rows=1, cols=len(titles), subplot_titles=titles, horizontal_spacing=0.018
    )
    for col, step in enumerate(CAPTURE_STEPS, start=1):
        add_board(
            fig,
            snapshots[step][0].numpy(),
            regions[puzzle_index],
            1,
            col,
            queens=step == FLOW_STEPS,
        )
    add_board(
        fig,
        target_tokens[puzzle_index],
        regions[puzzle_index],
        1,
        len(titles),
        queens=True,
    )
    source = puzzles["source_names"][puzzle_index]
    status = "SOLVED" if solved else "UNSOLVED"
    style_boards(
        fig, f"{source} · test puzzle {puzzle_index} · attempt {attempt + 1} · {status}"
    ).show(config=PLOT_CONFIG)

IntProgress(value=0, description='Trajectories', layout=Layout(width='100%'), max=600, style=ProgressStyle(des…

## 7. How often is a held-out puzzle solved?

Run independent stochastic trajectories from each selected test puzzle's full coloring.

- **Solve probability:** average success over puzzles and attempts.
- **Solved at least once:** fraction of puzzles solved by any of the repeated attempts.
- **Exact match:** all output tokens equal the supplied target.
- Strict scoring checks every Queens rule on the raw output. Checking a proposed answer is linear-time; it does not search for solutions or verify uniqueness.

The 95% interval bootstraps whole puzzles to preserve the grouping of repeated attempts. Source-specific rates separate externally collected and synthetic puzzles.

The retry curve shows the probability of finding at least one solution within k attempts. It averages over all subsets of k recorded attempts for each puzzle, then across puzzles. The shaded band is a 95% puzzle-bootstrap interval. The curve stops at the number of attempts actually sampled.

The orange dashed reference assumes the same single-attempt probability $\bar p$ for every puzzle and shows $1-(1-\bar p)^k$. The blue curve estimates success separately for each puzzle before averaging.


In [14]:
N_EVAL_PUZZLES = 256  # None evaluates every held-out puzzle.
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
evaluation_rng = np.random.default_rng(EVAL_SEED)
evaluation_indices = evaluation_rng.permutation(splits["test"])
if N_EVAL_PUZZLES is not None:
    if N_EVAL_PUZZLES < 1:
        raise ValueError("N_EVAL_PUZZLES must be positive or None.")
    evaluation_indices = evaluation_indices[:N_EVAL_PUZZLES]
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
evaluation_progress = widgets.IntProgress(
    value=0,
    max=len(flat_indices) * FLOW_STEPS,
    description="Evaluation",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
predicted_batches = []
for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
    batch_indices = flat_indices[start : start + EVAL_BATCH_SIZE]
    offset = start * FLOW_STEPS

    def update_evaluation(done, total, offset=offset, count=len(batch_indices)):
        evaluation_progress.value = offset + done * count

    prediction, _ = sample_queens(
        model,
        regions[batch_indices],
        steps=FLOW_STEPS,
        seed=EVAL_SEED + start,
        progress=update_evaluation,
    )
    predicted_batches.append(prediction.numpy())
    evaluation_status.value = (
        f"{start + len(batch_indices):,} / {len(flat_indices):,} trajectories"
    )
evaluation_progress.bar_style = "success"

predicted_tokens = np.concatenate(predicted_batches, axis=0)
checks = solution_checks(regions[flat_indices], predicted_tokens)
successes = checks["solved"].reshape(len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
exact_matches = np.all(
    predicted_tokens == target_tokens[flat_indices], axis=(-2, -1)
).reshape(successes.shape)
per_puzzle_success = successes.mean(axis=1)
solve_probability = float(per_puzzle_success.mean())
bootstrap_rng = np.random.default_rng(EVAL_SEED + 1)
bootstrap_means = per_puzzle_success[
    bootstrap_rng.integers(
        0, len(per_puzzle_success), size=(3000, len(per_puzzle_success))
    )
].mean(axis=1)
lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])
at_least_once = float(successes.any(axis=1).mean())
display(
    HTML(
        f"<b>Solved: {100 * solve_probability:.2f}%</b>"
        f" · 95% puzzle-bootstrap interval [{100 * lower:.2f}%, {100 * upper:.2f}%]<br>"
        f"Exact target match: {100 * exact_matches.mean():.2f}% · "
        f"Solved at least once in {ATTEMPTS_PER_PUZZLE} attempts: {100 * at_least_once:.2f}%<br>"
        f"{len(evaluation_indices)} unseen puzzles · {len(flat_indices):,} trajectories · "
        f"{FLOW_STEPS} steps · mean queen count {checks['queen_count'].mean():.2f}"
    )
)
evaluation_results = dict(
    dataset_indices=evaluation_indices,
    predicted_tokens=predicted_tokens.reshape(
        len(evaluation_indices), ATTEMPTS_PER_PUZZLE, BOARD_SIZE, BOARD_SIZE
    ),
    successes=successes,
    exact_matches=exact_matches,
    per_puzzle_success=per_puzzle_success,
    steps=FLOW_STEPS,
)

IntProgress(value=0, description='Evaluation', layout=Layout(width='100%'), max=819200, style=ProgressStyle(de…

HTML(value='')

In [15]:
source_names = puzzles["source_names"][evaluation_indices]
names = ["All test puzzles"] + sorted(set(source_names.tolist()))
means, lows, highs, sample_counts = [], [], [], []
for name in names:
    values = (
        per_puzzle_success
        if name == "All test puzzles"
        else per_puzzle_success[source_names == name]
    )
    draw = bootstrap_rng.integers(0, len(values), size=(3000, len(values)))
    interval = np.quantile(values[draw].mean(axis=1), [0.025, 0.975])
    means.append(100 * float(values.mean()))
    lows.append(100 * float(interval[0]))
    highs.append(100 * float(interval[1]))
    sample_counts.append(len(values))

# For a puzzle with c successes in n recorded attempts, average over all
# subsets of k attempts: pass@k = 1 - C(n-c, k) / C(n, k).
# This uses the actual observed attempts and does not extrapolate beyond n.
attempt_count = successes.shape[1]
attempt_counts = np.arange(attempt_count + 1)
per_puzzle_pass_at_k = np.zeros((len(successes), attempt_count + 1))
no_solution_probability = np.ones(len(successes))
failed_attempts = attempt_count - successes.sum(axis=1)
for k in range(1, attempt_count + 1):
    no_solution_probability *= np.maximum(failed_attempts - k + 1, 0) / (
        attempt_count - k + 1
    )
    per_puzzle_pass_at_k[:, k] = 1.0 - no_solution_probability
pass_at_k = per_puzzle_pass_at_k.mean(axis=0)
mean_single_attempt = float(successes.mean())
mean_probability_pass_at_k = 1.0 - (1.0 - mean_single_attempt) ** attempt_counts

# Resample whole puzzles, keeping their repeated attempts together.
# Bootstrap weights avoid allocating a (bootstrap, puzzles, attempts) array.
retry_rng = np.random.default_rng(EVAL_SEED + 2)
retry_bootstrap_weights = retry_rng.multinomial(
    len(successes),
    np.full(len(successes), 1.0 / len(successes)),
    size=3000,
)
retry_bootstrap = retry_bootstrap_weights @ per_puzzle_pass_at_k / len(successes)
pass_at_k_lower, pass_at_k_upper = np.quantile(
    retry_bootstrap,
    [0.025, 0.975],
    axis=0,
)

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    subplot_titles=[
        "Solve probability - 95% intervals",
        "At least one solution vs number of attempts",
    ],
)
fig.add_trace(
    go.Bar(
        x=[f"{name}<br>n = {count}" for name, count in zip(names, sample_counts)],
        y=means,
        marker_color=["#4F6BED"] + ["#6CAEA0"] * (len(names) - 1),
        error_y=dict(
            type="data",
            symmetric=False,
            array=np.maximum(0, np.asarray(highs) - means),
            arrayminus=np.maximum(0, np.asarray(means) - lows),
        ),
        hovertemplate="%{x}<br>solved %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_lower,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(79,107,237,0.16)",
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k,
        mode="lines+markers",
        line=dict(color="#4F6BED", width=3),
        marker=dict(size=7),
        customdata=np.column_stack((100 * pass_at_k_lower, 100 * pass_at_k_upper)),
        hovertemplate=(
            "Attempts: %{x}<br>At least one solution: %{y:.2f}%"
            "<br>95% interval: [%{customdata[0]:.2f}%, %{customdata[1]:.2f}%]"
            "<extra></extra>"
        ),
        name="Per-puzzle estimate",
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * mean_probability_pass_at_k,
        mode="lines+markers",
        name="From mean success probability",
        line=dict(color="#D88437", width=3, dash="dash"),
        marker=dict(size=6),
        hovertemplate=(
            "Attempts: %{x}<br>From mean probability: %{y:.2f}%" "<extra></extra>"
        ),
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.update_yaxes(title_text="Solved trajectories (%)", range=[0, 105], row=1, col=1)
fig.update_yaxes(
    title_text="Probability of at least one solution (%)",
    rangemode="tozero",
    row=1,
    col=2,
)
fig.update_xaxes(
    title_text="Number of attempts",
    dtick=max(1, attempt_count // 10),
    range=[0, attempt_count],
    row=1,
    col=2,
)
fig.update_layout(
    title=dict(text=f"Queens DFM - held-out results - {FLOW_STEPS} steps", x=0),
    template="plotly_white",
    autosize=True,
    height=520,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.28, yanchor="top"),
    bargap=0.2,
    margin=dict(l=65, r=20, t=100, b=140),
    font=dict(family="Arial, sans-serif", size=16),
)
fig.show(config=PLOT_CONFIG)
evaluation_results.update(
    mean_single_attempt=mean_single_attempt,
    mean_probability_pass_at_k=mean_probability_pass_at_k,
    attempt_counts=attempt_counts,
    pass_at_k=pass_at_k,
    pass_at_k_lower=pass_at_k_lower,
    pass_at_k_upper=pass_at_k_upper,
)
np.savez_compressed(
    ARTIFACTS / "queens7_dfm_evaluation.npz",
    **evaluation_results,
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    source_names=source_names,
    evaluation_seed=EVAL_SEED,
)

### 7.1 From retry success to puzzle difficulty

The orange dashed curve uses the mean single-attempt success probability $\bar p$:

$$
P_{\mathrm{mean}}(k)=1-(1-\bar p)^k.
$$

This assumes every puzzle has the same success probability. For puzzles with individual probabilities $p_i$, independent retries on the same puzzle instead give

$$
P_{\mathrm{puzzles}}(k)
=\mathbb E_i\!\left[1-(1-p_i)^k\right].
$$

For $k\ge2$, the function $1-(1-p)^k$ is concave, so

$$
\mathbb E_i\!\left[1-(1-p_i)^k\right]
\le 1-\left(1-\mathbb E_i[p_i]\right)^k.
$$

Some puzzles are already relatively easy and are often solved repeatedly; others remain difficult across retries. Pooling them into one mean success probability overestimates the benefit of repeated attempts when their probabilities differ.

A gap between the orange and blue curves therefore motivates a study of **different puzzle difficulties for this model and sampler**. We estimate those differences from 32 independent attempts per puzzle; finite sampling also contributes uncertainty.


### 7.2 Solvability study

Estimate each held-out puzzle's solvability as its fraction of successful attempts:

$$
\widehat p_i=\frac{\text{successful attempts for puzzle }i}{32}.
$$

Plot a continuous, boundary-corrected kernel density of these estimates. The x-axis is **probability to be solved**; the y-axis is **% of puzzles on a logarithmic scale**, expressed per one percentage point of x so that the area under the density equals 100%.

The smoothing bandwidth defaults to one attempt's resolution ($100/32=3.125$ percentage points). This smooths the measured probabilities; it does not increase the information in 32 attempts. Zero observed successes means no successful attempt was observed for that puzzle. Very small density tails below 0.001% per percentage point are hidden on the log plot.

Rerun the evaluation cell with 32 attempts before this study. The three marked probabilities correspond to the example puzzles below.


In [16]:
study_successes = np.asarray(successes, dtype=bool)
study_attempts = study_successes.shape[1]
if study_attempts < 32:
    raise ValueError(
        "Solvability study needs at least 32 recorded attempts per puzzle. "
        "Rerun the evaluation cell with ATTEMPTS_PER_PUZZLE = 32."
    )
study_rates = study_successes.mean(axis=1)
study_probabilities_percent = 100.0 * study_rates

SOLVABILITY_BANDWIDTH_PP = 100.0 / study_attempts
SOLVABILITY_DENSITY_FLOOR = 0.001
if SOLVABILITY_BANDWIDTH_PP <= 0 or SOLVABILITY_DENSITY_FLOOR <= 0:
    raise ValueError("Bandwidth and the log-display density floor must be positive.")
solvability_grid = np.linspace(0.0, 100.0, 2001)
bandwidth = float(SOLVABILITY_BANDWIDTH_PP)
distance = (
    solvability_grid[:, None] - study_probabilities_percent[None, :]
) / bandwidth
left_reflection = (
    solvability_grid[:, None] + study_probabilities_percent[None, :]
) / bandwidth
right_reflection = (
    solvability_grid[:, None] - (200.0 - study_probabilities_percent[None, :])
) / bandwidth
solvability_density = (
    np.exp(-0.5 * distance**2)
    + np.exp(-0.5 * left_reflection**2)
    + np.exp(-0.5 * right_reflection**2)
).mean(axis=1) / (bandwidth * np.sqrt(2.0 * np.pi))
# Reflection retains boundary mass at 0% and 100%; normalize the sampled grid.
density_area = np.sum(
    0.5
    * (solvability_density[:-1] + solvability_density[1:])
    * np.diff(solvability_grid)
)
solvability_density_percent = 100.0 * solvability_density / density_area
visible_density = np.where(
    solvability_density_percent >= SOLVABILITY_DENSITY_FLOOR,
    solvability_density_percent,
    np.nan,
)

# Pick distinct examples: highest observed rate, then approximately half
# the previous selected example's rate. Use actual available measurements.
study_example_positions = []
study_example_targets = []
remaining_positions = np.arange(len(study_rates))
previous_rate = None
for _ in range(min(3, len(study_rates))):
    if previous_rate is None:
        target_rate = float(study_rates.max())
        candidates = remaining_positions
    else:
        target_rate = previous_rate / 2.0
        lower_candidates = remaining_positions[
            study_rates[remaining_positions] < previous_rate
        ]
        candidates = lower_candidates if len(lower_candidates) else remaining_positions
    differences = np.abs(study_rates[candidates] - target_rate)
    position = int(candidates[np.argmin(differences)])
    study_example_positions.append(position)
    study_example_targets.append(target_rate)
    previous_rate = float(study_rates[position])
    remaining_positions = remaining_positions[remaining_positions != position]

study_example_colors = ["#D88437", "#2F9D88", "#9B6BD3"]
solvability_fig = go.Figure(
    go.Scatter(
        x=solvability_grid,
        y=visible_density,
        mode="lines",
        line=dict(color="#4F6BED", width=3),
        name="Smoothed measured solvability",
        showlegend=False,
        hovertemplate=(
            "Probability to be solved: %{x:.2f}%"
            "<br>Puzzles per 1 percentage point: %{y:.4f}%<extra></extra>"
        ),
    )
)
for rank, position in enumerate(study_example_positions):
    probability_percent = float(study_probabilities_percent[position])
    density_at_probability = float(
        np.interp(probability_percent, solvability_grid, solvability_density_percent)
    )
    solvability_fig.add_vline(
        x=probability_percent,
        line=dict(color=study_example_colors[rank], width=1.3, dash="dot"),
    )
    solvability_fig.add_trace(
        go.Scatter(
            x=[probability_percent],
            y=[density_at_probability],
            mode="markers",
            marker=dict(color=study_example_colors[rank], size=10),
            name=f"Example {rank + 1}: {probability_percent:.2f}%",
            hovertemplate=(
                f"Example {rank + 1}<br>Measured solvability: {probability_percent:.2f}%"
                "<extra></extra>"
            ),
        )
    )
solvability_fig.update_layout(
    title=dict(
        text=f"Solvability study - {len(study_rates)} puzzles, "
        f"{study_attempts} attempts each",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=500,
    xaxis_title="Probability to be solved (%)",
    yaxis_title="% of puzzles per 1 percentage point (log scale)",
    yaxis_type="log",
    legend=dict(orientation="h", x=0, y=-0.24),
    margin=dict(l=85, r=25, t=85, b=115),
    font=dict(family="Arial, sans-serif", size=16),
)
solvability_fig.update_xaxes(range=[0, 100], dtick=10)
solvability_fig.update_yaxes(
    range=[
        np.log10(SOLVABILITY_DENSITY_FLOOR),
        np.ceil(np.log10(solvability_density_percent.max())) + 0.1,
    ]
)
solvability_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"Measured resolution: {100 / study_attempts:.3f} percentage points per success."
        f" Smoothing bandwidth: {bandwidth:.3f} percentage points.<br>"
        f"No observed success: {np.mean(study_rates == 0):.1%} of puzzles."
    )
)

np.savez_compressed(
    ARTIFACTS / "queens7_solvability_study.npz",
    dataset_indices=evaluation_indices,
    successes=study_successes,
    per_puzzle_probability=study_rates,
    grid_probability_percent=solvability_grid,
    density_percent_per_percentage_point=solvability_density_percent,
    bandwidth_percentage_points=bandwidth,
    example_positions=np.asarray(study_example_positions),
    dataset_sha256=puzzles["metadata"]["data_sha256"],
)

### 7.3 Example puzzles by solvability

Start with the most frequently solved evaluated puzzle. Then select a different puzzle whose measured success probability is closest to half of the previous example's probability, preferring a strictly lower rate when one is available.

The examples appear in decreasing solvability order. Titles show the **actual measured probability and success count**, since exact 100%, 50% and 25% examples may not exist. The queens shown are the supplied ground-truth solution on the original coloring.


In [17]:
example_titles = []
for rank, position in enumerate(study_example_positions):
    count = int(study_successes[position].sum())
    probability_percent = float(study_probabilities_percent[position])
    example_titles.append(
        f"Example {rank + 1}: {probability_percent:.2f}%"
        f"<br>{count}/{study_attempts} successful attempts"
    )
study_examples_fig = make_subplots(
    rows=1,
    cols=len(study_example_positions),
    subplot_titles=example_titles,
    horizontal_spacing=0.065,
)
selected_study_examples = []
for col, position in enumerate(study_example_positions, start=1):
    puzzle_index = int(evaluation_indices[position])
    add_board(study_examples_fig, regions[puzzle_index], regions[puzzle_index], 1, col)
    queen_rows, queen_cols = np.where(queen_masks[puzzle_index])
    study_examples_fig.add_trace(
        go.Scatter(
            x=queen_cols,
            y=queen_rows,
            mode="text",
            text=[chr(0x265B)] * len(queen_rows),
            textfont=dict(size=25, color="#172033"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    selected_study_examples.append(
        {
            "dataset_index": puzzle_index,
            "id": puzzles["source_ids"][puzzle_index],
            "source": puzzles["source_names"][puzzle_index],
            "successes": int(study_successes[position].sum()),
            "attempts": study_attempts,
            "probability": float(study_rates[position]),
            "target_probability": float(study_example_targets[col - 1]),
        }
    )
style_boards(
    study_examples_fig,
    "Solvability study - example puzzles",
    rows=1,
)
study_examples_fig.update_layout(height=410, margin=dict(l=10, r=10, t=120, b=20))
study_examples_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        "<br>".join(
            f"<b>Example {rank + 1}</b>: "
            f"{html.escape(str(example['source']))} / {html.escape(str(example['id']))}"
            for rank, example in enumerate(selected_study_examples)
        )
    )
)